In [17]:
import pandas as pd
from fastparquet import write
from fastparquet import ParquetFile
from sklearn.preprocessing import OrdinalEncoder
from pathlib import Path
import numpy as np
import os

In [18]:
data_pipeline = "label_encode"
input_pipeline = "nums_to_cats"

In [19]:
if os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
    data_path = "../../kaggle/input/datasets/abhinavneelam/smartphone-addiction/data/"
    output_path = "/kaggle/working/"
else:
    data_path = "../../data/"
    output_path = "../../"

In [20]:
experiment_path = Path(data_path) / f"{data_pipeline}"
experiment_path.mkdir(parents=True, exist_ok=True)

In [21]:
ss = pd.read_csv("../../data/raw/sample_submission.csv")
target_column = ss.columns[-1]
target_column

'addicted_label'

In [22]:
X = ParquetFile(Path(data_path) / f"{input_pipeline}/train.parq").to_pandas()
X_test = ParquetFile(Path(data_path) / f"{input_pipeline}/test.parq").to_pandas()

X.info()

<class 'pandas.DataFrame'>
RangeIndex: 691369 entries, 0 to 691368
Data columns (total 11 columns):
 #   Column                       Non-Null Count   Dtype   
---  ------                       --------------   -----   
 0   age_cat                      662440 non-null  category
 1   daily_screen_time_hours_cat  595515 non-null  category
 2   social_media_hours_cat       557374 non-null  category
 3   gaming_hours_cat             564548 non-null  category
 4   work_study_hours_cat         639851 non-null  category
 5   sleep_hours_cat              646889 non-null  category
 6   notifications_per_day_cat    623785 non-null  category
 7   app_opens_per_day_cat        610659 non-null  category
 8   weekend_screen_time_cat      579306 non-null  category
 9   stress_level_cat             636221 non-null  category
 10  academic_work_impact_cat     647145 non-null  category
dtypes: category(11)
memory usage: 12.6 MB


In [23]:
X.head()

,age_cat,daily_screen_time_hours_cat,social_media_hours_cat,gaming_hours_cat,work_study_hours_cat,sleep_hours_cat,notifications_per_day_cat,app_opens_per_day_cat,weekend_screen_time_cat,stress_level_cat,academic_work_impact_cat
0,24.0,NaN,1.83,1.59,2.11,7.46,122.0,38.0,8.63,1.0,0.0
1,19.0,5.97,1.08,NaN,3.03,8.22,76.0,19.0,NaN,1.0,0.0
2,18.0,5.09,NaN,NaN,NaN,6.25,134.0,60.0,7.47,0.0,1.0
3,21.0,6.42,1.26,1.42,3.36,8.85,112.0,94.0,8.66,0.0,NaN
4,26.0,11.20,1.87,2.81,1.95,5.25,NaN,NaN,13.39,1.0,0.0


In [24]:
drop_columns = X.columns
cat_columns = X.select_dtypes(include=['category']).columns.to_list()

encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
encoder.fit(X[cat_columns])

X_encoded = encoder.transform(X[cat_columns])
X_test_encoded = encoder.transform(X_test[cat_columns])

new_cols = [col + '_le' for col in cat_columns]
X = pd.DataFrame(X_encoded, columns=new_cols, index=X.index)
X_test = pd.DataFrame(X_test_encoded, columns=new_cols, index=X_test.index)

X.info()

<class 'pandas.DataFrame'>
RangeIndex: 691369 entries, 0 to 691368
Data columns (total 11 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   age_cat_le                      662440 non-null  float64
 1   daily_screen_time_hours_cat_le  595515 non-null  float64
 2   social_media_hours_cat_le       557374 non-null  float64
 3   gaming_hours_cat_le             564548 non-null  float64
 4   work_study_hours_cat_le         639851 non-null  float64
 5   sleep_hours_cat_le              646889 non-null  float64
 6   notifications_per_day_cat_le    623785 non-null  float64
 7   app_opens_per_day_cat_le        610659 non-null  float64
 8   weekend_screen_time_cat_le      579306 non-null  float64
 9   stress_level_cat_le             636221 non-null  float64
 10  academic_work_impact_cat_le     647145 non-null  float64
dtypes: float64(11)
memory usage: 58.0 MB


In [25]:
write(experiment_path / f"train.parq", X)
write(experiment_path / f"test.parq", X_test)